# DDI Network Visualization — Gephi Data Preparation

**Purpose:** Generate node and edge list files for import into Gephi to create publication-quality drug-drug interaction network graphs.

**Input:** FAERS DDI signal files from the main analysis  
**Output:** `gephi_nodes.csv`, `gephi_edges.csv` (+ top 50/100 filtered versions)

---

### Gephi Import Instructions (after running this notebook):
1. Open Gephi → File → New Project
2. File → Import Spreadsheet → select `gephi_edges_top100.csv` → Import as **Edge Table**
3. File → Import Spreadsheet → select `gephi_nodes_top100.csv` → Import as **Node Table** → **Append** to existing workspace
4. Layout → **ForceAtlas2** → Run (let it stabilize ~30 seconds, then stop)
5. Appearance → Nodes → Size → Ranking → **FREQ** → Apply
6. Appearance → Nodes → Color → Partition → **DRUG_CLASS** → Apply
7. Appearance → Edges → Color → Ranking → **ROR** → Apply (use gradient: green → yellow → red)
8. Preview → Refresh → Export as PNG/SVG for the paper

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

## Configuration

Adjust these thresholds to control network size.

In [2]:
# ---- CONFIGURATION ----
# Adjust these thresholds to control network size
ROR_THRESHOLD = 2.0        # Minimum ROR for inclusion
MIN_SAMPLE_SIZE = 50       # Minimum N_EXPOSED for inclusion
MAX_EDGES = 200            # Top N edges by ROR (keeps Gephi manageable)

print(f"ROR threshold: {ROR_THRESHOLD}")
print(f"Min sample size: {MIN_SAMPLE_SIZE}")
print(f"Max edges: {MAX_EDGES}")

ROR threshold: 2.0
Min sample size: 50
Max edges: 200


## Load and Filter Signal Data

In [3]:
# ---- LOAD SIGNAL DATA ----
try:
    signals = pd.read_csv("FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv")
    print(f"Loaded high-confidence signals: {len(signals):,}")
    source = "HIGH_CONFIDENCE"
except FileNotFoundError:
    signals = pd.read_csv("FAERS_DDI_SIGNALS.csv")
    print(f"Loaded all signals: {len(signals):,}")
    source = "ALL_SIGNALS"

# ---- FILTER SIGNALS ----
print(f"\nFiltering: ROR > {ROR_THRESHOLD}, N >= {MIN_SAMPLE_SIZE}")
filtered = signals[
    (signals['ROR'] > ROR_THRESHOLD) &
    (signals['N_EXPOSED'] >= MIN_SAMPLE_SIZE)
].copy()

print(f"After filtering: {len(filtered):,} signals")

# Take top N by ROR
filtered = filtered.nlargest(MAX_EDGES, 'ROR')
print(f"Top {MAX_EDGES} edges selected")

Loaded high-confidence signals: 5,501

Filtering: ROR > 2.0, N >= 50
After filtering: 5,501 signals
Top 200 edges selected


## Drug Class Mapping

Map drugs to high-risk therapeutic classes for color-coding nodes in Gephi.

In [4]:
# ---- HIGH-RISK DRUG CLASS MAPPING ----
high_risk_classes = {
    'ANTICOAGULANTS': ['WARFARIN', 'HEPARIN', 'ENOXAPARIN', 'RIVAROXABAN', 'APIXABAN',
                       'DABIGATRAN', 'EDOXABAN', 'FONDAPARINUX', 'COUMADIN', 'XARELTO',
                       'ELIQUIS', 'PRADAXA', 'LOVENOX'],
    'ANTIPLATELET': ['ASPIRIN', 'CLOPIDOGREL', 'PLAVIX', 'TICAGRELOR', 'PRASUGREL',
                     'BRILINTA', 'EFFIENT', 'DIPYRIDAMOLE'],
    'IMMUNOSUPPRESSANTS': ['TACROLIMUS', 'CYCLOSPORINE', 'MYCOPHENOLATE', 'AZATHIOPRINE',
                           'SIROLIMUS', 'EVEROLIMUS', 'PROGRAF', 'CELLCEPT', 'IMURAN'],
    'BIOLOGICS': ['ADALIMUMAB', 'INFLIXIMAB', 'ETANERCEPT', 'RITUXIMAB', 'TOCILIZUMAB',
                  'HUMIRA', 'REMICADE', 'ENBREL', 'ACTEMRA', 'DUPIXENT', 'VEDOLIZUMAB'],
    'OPIOIDS': ['OXYCODONE', 'HYDROCODONE', 'MORPHINE', 'FENTANYL', 'TRAMADOL',
                'CODEINE', 'METHADONE', 'BUPRENORPHINE', 'HYDROMORPHONE'],
    'ANTINEOPLASTICS': ['METHOTREXATE', 'CYCLOPHOSPHAMIDE', 'DOXORUBICIN', 'PACLITAXEL',
                        'CARBOPLATIN', 'CISPLATIN', 'FLUOROURACIL', 'VINCRISTINE',
                        'IMATINIB', 'PEMBROLIZUMAB', 'NIVOLUMAB'],
    'NARROW_THERAPEUTIC_INDEX': ['DIGOXIN', 'LITHIUM', 'PHENYTOIN', 'CARBAMAZEPINE',
                                  'THEOPHYLLINE', 'VALPROIC', 'GENTAMICIN', 'VANCOMYCIN']
}

def get_drug_class(drug_name):
    drug_upper = drug_name.upper()
    for class_name, drugs in high_risk_classes.items():
        for d in drugs:
            if d in drug_upper:
                return class_name
    return 'OTHER'

print(f"Drug classes defined: {len(high_risk_classes)}")
print("Classes:", list(high_risk_classes.keys()))

Drug classes defined: 7
Classes: ['ANTICOAGULANTS', 'ANTIPLATELET', 'IMMUNOSUPPRESSANTS', 'BIOLOGICS', 'OPIOIDS', 'ANTINEOPLASTICS', 'NARROW_THERAPEUTIC_INDEX']


## Create Edge List

In [5]:
# ---- CREATE EDGE LIST ----
print("Creating edge list...")

edges = []
for _, row in filtered.iterrows():
    pair = row['PAIR']
    drugs = pair.split(' + ')
    if len(drugs) == 2:
        edges.append({
            'Source': drugs[0].strip(),
            'Target': drugs[1].strip(),
            'Weight': round(row['ROR'], 2),
            'ROR': round(row['ROR'], 2),
            'N_EXPOSED': int(row['N_EXPOSED']),
            'A_SERIOUS': int(row['A_SERIOUS_EXPOSED']),
            'Type': 'Undirected',
            'Label': pair
        })

edges_df = pd.DataFrame(edges)
print(f"Edges created: {len(edges_df):,}")

# ---- ROR SEVERITY CATEGORIES ----
edges_df['SEVERITY'] = pd.cut(
    edges_df['ROR'],
    bins=[0, 3, 5, 10, float('inf')],
    labels=['MODERATE (2-3)', 'STRONG (3-5)', 'VERY_STRONG (5-10)', 'EXTREME (>10)']
)

print(f"\nEdge ROR range: {edges_df['ROR'].min():.1f} - {edges_df['ROR'].max():.1f}")
print(f"Median ROR: {edges_df['ROR'].median():.1f}")
print(f"\nEdges by severity:")
print(edges_df['SEVERITY'].value_counts())

Creating edge list...
Edges created: 200

Edge ROR range: 7.0 - 8.7
Median ROR: 7.7

Edges by severity:
SEVERITY
VERY_STRONG (5-10)    200
MODERATE (2-3)          0
STRONG (3-5)            0
EXTREME (>10)           0
Name: count, dtype: int64


## Create Node List

In [6]:
# ---- CREATE NODE LIST ----
print("Creating node list...")

# Count drug frequency (degree in the network)
all_drugs = edges_df['Source'].tolist() + edges_df['Target'].tolist()
drug_freq = Counter(all_drugs)

# Build node table
nodes = []
for drug, freq in drug_freq.items():
    drug_class = get_drug_class(drug)

    # Calculate average ROR for edges involving this drug
    drug_edges = edges_df[(edges_df['Source'] == drug) | (edges_df['Target'] == drug)]
    avg_ror = drug_edges['ROR'].mean() if len(drug_edges) > 0 else 0
    max_ror = drug_edges['ROR'].max() if len(drug_edges) > 0 else 0

    nodes.append({
        'Id': drug,
        'Label': drug,
        'FREQ': freq,                    # Number of interactions (degree)
        'DRUG_CLASS': drug_class,         # For color coding in Gephi
        'AVG_ROR': round(avg_ror, 2),     # Average interaction strength
        'MAX_ROR': round(max_ror, 2),     # Strongest interaction
    })

nodes_df = pd.DataFrame(nodes)

print(f"Nodes created: {len(nodes_df):,}")
print(f"\nDrugs by class:")
print(nodes_df['DRUG_CLASS'].value_counts())

print(f"\nTop 20 most connected drugs:")
print(nodes_df.nlargest(20, 'FREQ')[['Label', 'FREQ', 'DRUG_CLASS', 'AVG_ROR']].to_string(index=False))

Creating node list...
Nodes created: 131

Drugs by class:
DRUG_CLASS
OTHER                       106
BIOLOGICS                     8
OPIOIDS                       6
ANTINEOPLASTICS               5
IMMUNOSUPPRESSANTS            3
ANTICOAGULANTS                2
NARROW_THERAPEUTIC_INDEX      1
Name: count, dtype: int64

Top 20 most connected drugs:
                                                 Label  FREQ DRUG_CLASS  AVG_ROR
                                     DICLOFENAC SODIUM    12      OTHER     7.56
CETIRIZINE HYDROCHLORIDE\PSEUDOEPHEDRINE HYDROCHLORIDE    11      OTHER     7.84
                                         SULFASALAZINE    11      OTHER     7.60
                                           LEFLUNOMIDE    10      OTHER     7.56
                                               FOSAMAX    10      OTHER     7.79
                                            DICLOFENAC    10      OTHER     8.04
                                        DESOXIMETASONE    10      OTHER     7.83
   

## Save Gephi Files

Three versions:
- **Full network** — all filtered edges
- **Top 100** — main paper figure
- **Top 50** — cleaner/simpler figure

In [7]:
# ---- SAVE FULL NETWORK ----
nodes_df.to_csv("gephi_nodes.csv", index=False)
edges_df.to_csv("gephi_edges.csv", index=False)
print(f"Full network: gephi_nodes.csv ({len(nodes_df)} nodes) + gephi_edges.csv ({len(edges_df)} edges)")

# ---- TOP 100 (MAIN FIGURE) ----
top100 = edges_df.nlargest(100, 'ROR')
top100_drugs = set(top100['Source'].tolist() + top100['Target'].tolist())
top100_nodes = nodes_df[nodes_df['Id'].isin(top100_drugs)]

top100_nodes.to_csv("gephi_nodes_top100.csv", index=False)
top100.to_csv("gephi_edges_top100.csv", index=False)
print(f"\nTop 100: gephi_nodes_top100.csv ({len(top100_nodes)} nodes) + gephi_edges_top100.csv (100 edges)")

# ---- TOP 50 (CLEAN FIGURE) ----
top50 = edges_df.nlargest(50, 'ROR')
top50_drugs = set(top50['Source'].tolist() + top50['Target'].tolist())
top50_nodes = nodes_df[nodes_df['Id'].isin(top50_drugs)]

top50_nodes.to_csv("gephi_nodes_top50.csv", index=False)
top50.to_csv("gephi_edges_top50.csv", index=False)
print(f"\nTop 50:  gephi_nodes_top50.csv ({len(top50_nodes)} nodes) + gephi_edges_top50.csv (50 edges)")

Full network: gephi_nodes.csv (131 nodes) + gephi_edges.csv (200 edges)

Top 100: gephi_nodes_top100.csv (94 nodes) + gephi_edges_top100.csv (100 edges)

Top 50:  gephi_nodes_top50.csv (67 nodes) + gephi_edges_top50.csv (50 edges)


## Next Steps in Gephi

After importing, use these Gephi settings for a publication-quality figure:

**Layout:**
- ForceAtlas2 with Scaling = 10, Gravity = 1.0
- Let it run ~30-60 seconds until stable, then stop

**Node appearance:**
- **Size** → Ranking → FREQ (min: 10, max: 50) — bigger = more interactions
- **Color** → Partition → DRUG_CLASS — each therapeutic class gets a distinct color

**Edge appearance:**
- **Color** → Ranking → ROR — use a gradient (green → yellow → red)
- **Thickness** → Ranking → ROR or N_EXPOSED

**Labels:**
- Show node labels, adjust font size proportional to node size

**Export:**
- Preview tab → Refresh → Export as SVG (vector, editable) or PNG (raster, 300 DPI)
- Use SVG if submitting to a journal — it scales without pixelation